<a href="https://colab.research.google.com/github/juancis82/Juancis82/blob/main/diagnostico_conectividad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from collections import deque
from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Tuple

# ---------------------------------------------------------------------------
# Configuración
# ---------------------------------------------------------------------------
VENTANA = 20          # mediciones recordadas por usuario
MIN_PICOS = 6         # picos en la ventana para marcar intermitencia
PICO_PERDIDA = 1.0    # % de pérdida que cuenta como pico
MODO_INTERACTIVO = False

# ---------------------------------------------------------------------------
# Hechos de dominio
# ---------------------------------------------------------------------------
# (aplicación, síntoma): (umbral, comparación, fuente)
# fuente: norma = ITU-T G.114 / Cisco, guia = guías de diseño Wi-Fi, propio = criterio de diseño
UMBRALES = {
    ('Videollamada', 'Jitter'):      (30, '>', 'norma'),
    ('Videollamada', 'Perdida'):     (1, '>', 'norma'),
    ('Videollamada', 'Latencia'):    (300, '>', 'norma'),
    ('Videollamada', 'SenalDebil'):  (-67, '<', 'guia'),
    ('Videollamada', 'Bufferbloat'): (100, '>', 'propio'),
    ('Videollamada', 'AnchoBajo'):   (0.5, '<', 'propio'),
    ('Juegos', 'Latencia'):          (100, '>', 'propio'),
    ('Juegos', 'Jitter'):            (30, '>', 'propio'),
    ('Juegos', 'Perdida'):           (1, '>', 'propio'),
    ('Juegos', 'SenalDebil'):        (-67, '<', 'guia'),
    ('Juegos', 'Bufferbloat'):       (100, '>', 'propio'),
    ('Streaming', 'AnchoBajo'):      (0.5, '<', 'propio'),
    ('Streaming', 'Perdida'):        (2, '>', 'propio'),
    ('Streaming', 'SenalDebil'):     (-67, '<', 'guia'),
    ('Navegacion', 'DNS'):           (100, '>', 'propio'),
    ('Navegacion', 'Latencia'):      (200, '>', 'propio'),
    ('Navegacion', 'AnchoBajo'):     (0.5, '<', 'propio'),
    ('Navegacion', 'SenalDebil'):    (-75, '<', 'guia'),
    ('*', 'LatGateway'):             (20, '>', 'propio'),
}

SINTOMAS = ('Jitter', 'Perdida', 'Latencia', 'SenalDebil', 'Bufferbloat', 'AnchoBajo', 'DNS')
PERCIBIDOS = ('Jitter', 'Perdida', 'Latencia', 'AnchoBajo', 'DNS')

REMEDIOS = {
    'WiFi': 'Acercarse al router, pasar a la banda de 5 GHz o conectar por cable',
    'LAN': 'Revisar cableado, reiniciar el router y verificar equipos que saturen la red local',
    'ISP': 'Reclamar al proveedor adjuntando las mediciones registradas',
    'Congestion': 'Activar QoS/SQM en el router o evitar el horario pico',
    'DNS': 'Cambiar el servidor DNS del router o del equipo',
}

UNIDAD = {'Jitter': ' ms', 'Perdida': ' %', 'Latencia': ' ms', 'SenalDebil': ' dBm',
          'Bufferbloat': ' ms', 'AnchoBajo': '', 'DNS': ' ms', 'LatGateway': ' ms'}
ETIQUETA = {'Jitter': 'jitter', 'Perdida': 'pérdida', 'Latencia': 'RTT internet',
            'SenalDebil': 'señal Wi-Fi', 'Bufferbloat': 'latencia bajo carga',
            'AnchoBajo': 'descarga/plan', 'DNS': 'tiempo DNS', 'LatGateway': 'RTT router'}
CAMPOS = {'Jitter': ['jitter'], 'Perdida': ['perdida'], 'Latencia': ['lat_internet'],
          'SenalDebil': ['conexion', 'senal_wifi'], 'Bufferbloat': ['incr_lat_carga'],
          'AnchoBajo': ['descarga', 'plan'], 'DNS': ['dns'], 'LatGateway': ['lat_gateway']}


def medida(m: str, med: Dict) -> Optional[float]:
    if m == 'AnchoBajo':
        if med.get('descarga') is None or not med.get('plan'):
            return None
        return med['descarga'] / med['plan']
    if m == 'SenalDebil':
        return med.get('senal_wifi') if med.get('conexion') == 'wifi' else None
    campo = {'Jitter': 'jitter', 'Perdida': 'perdida', 'Latencia': 'lat_internet',
             'Bufferbloat': 'incr_lat_carga', 'DNS': 'dns', 'LatGateway': 'lat_gateway'}[m]
    return med.get(campo)


def buscar_umbral(app: str, m: str):
    return UMBRALES.get((app, m)) or UMBRALES.get(('*', m))


# Módulo de adquisición: altera el conocimiento sin tocar el motor
def actualizar_umbral(app: str, metrica: str, valor: float, comparacion: str, fuente: str = 'propio'):
    if metrica not in UNIDAD:
        raise ValueError(f"Métrica desconocida: {metrica}")
    if comparacion not in ('>', '<'):
        raise ValueError("La comparación debe ser '>' o '<'")
    if fuente not in ('norma', 'guia', 'propio'):
        raise ValueError("Fuente inválida")
    UMBRALES[(app, metrica)] = (valor, comparacion, fuente)


# ---------------------------------------------------------------------------
# Reglas
# ---------------------------------------------------------------------------
@dataclass
class Regla:
    id: str
    estrato: int
    premisa: tuple
    conclusion: tuple
    cf: float
    justificacion: str


def sint(s):
    return ('Sintoma', s)


REGLAS: List[Regla] = [
    Regla('R1', 1, ('umbral', 'Jitter'), sint('Jitter'), 0.90,
          'La variación de latencia supera lo tolerable para la aplicación'),
    Regla('R2', 1, ('umbral', 'Perdida'), sint('Perdida'), 0.90,
          'La pérdida de paquetes supera lo tolerable para la aplicación'),
    Regla('R3', 1, ('umbral', 'Latencia'), sint('Latencia'), 0.85,
          'El tiempo de ida y vuelta a internet supera lo tolerable para la aplicación'),
    Regla('R4', 1, ('umbral', 'SenalDebil'), sint('SenalDebil'), 0.80,
          'La señal Wi-Fi está por debajo del mínimo para la aplicación'),
    Regla('R5', 1, ('umbral', 'Bufferbloat'), sint('Bufferbloat'), 0.75,
          'La latencia sube de forma excesiva cuando el enlace está cargado'),
    Regla('R6', 1, ('umbral', 'AnchoBajo'), sint('AnchoBajo'), 0.85,
          'La velocidad obtenida es inferior a la proporción esperada del plan'),
    Regla('R7', 1, ('umbral', 'DNS'), sint('DNS'), 0.70,
          'La resolución de nombres tarda más de lo aceptable'),
    Regla('R8', 2, ('y', [('hecho', sint('SenalDebil')),
                          ('o', [('hecho', sint(s)) for s in ('Jitter', 'Perdida', 'Latencia', 'AnchoBajo')])]),
          ('Causa', 'WiFi'), 0.85,
          'Señal débil acompañada de degradación: el problema está en el enlace inalámbrico'),
    Regla('R9', 2, ('y', [('umbral', 'LatGateway'), ('no_hecho', sint('SenalDebil'))]),
          ('Causa', 'LAN'), 0.70,
          'El router responde lento aun con buena señal: el problema está en la red local'),
    Regla('R10', 2, ('y', [('no_umbral', 'LatGateway'),
                           ('o', [('hecho', sint('Perdida')), ('hecho', sint('Latencia')),
                                  ('flag', 'intermitente')])]),
          ('Causa', 'ISP'), 0.80,
          'La red local responde bien y la falla aparece más allá del router: acceso del proveedor'),
    Regla('R11', 2, ('o', [('hecho', sint('Bufferbloat')),
                           ('y', [('horario', 'pico'), ('hecho', sint('AnchoBajo'))])]),
          ('Causa', 'Congestion'), 0.75,
          'Enlace saturado: latencia bajo carga elevada o baja velocidad en horario pico'),
    Regla('R12', 2, ('y', [('hecho', sint('DNS')), ('no_hecho', sint('Latencia'))]),
          ('Causa', 'DNS'), 0.80,
          'La conexión responde bien pero la resolución de nombres es lenta'),
] + [
    Regla('R13', 2, ('hecho', sint(m)), ('ExperienciaMala',), 0.85,
          f'El síntoma {m} afecta la aplicación en uso') for m in PERCIBIDOS
] + [
    Regla('R14', 3, ('y', [('no_hecho', sint(s)) for s in SINTOMAS]),
          ('Diagnostico', 'SinProblemas'), 0.90,
          'Ninguna medición supera los umbrales de la aplicación'),
]


# ---------------------------------------------------------------------------
# Motor de inferencia
# ---------------------------------------------------------------------------
class Contexto:
    def __init__(self, usuario, app, med, picos, total):
        self.usuario, self.app, self.med = usuario, app, med
        self.picos, self.total = picos, total
        self.intermitente = picos >= MIN_PICOS
        self.consultados, self.faltantes = set(), set()


def fmt(h: tuple) -> str:
    return h[0] if len(h) == 1 else f"{h[0]}({', '.join(h[1:])})"


def combinar(a: float, b: float) -> float:
    return a + b * (1 - a)


def evaluar(p: tuple, ctx: Contexto, consulta: Callable):
    """Devuelve (cumple, cf, evidencia)."""
    t = p[0]
    if t == 'y':
        cfs, evid = [], []
        for q in p[1]:
            ok, cf, ev = evaluar(q, ctx, consulta)
            if not ok:
                return False, 0.0, ev
            cfs.append(cf)
            evid += ev
        return True, min(cfs), evid
    if t == 'o':
        res = [evaluar(q, ctx, consulta) for q in p[1]]
        ciertos = [r for r in res if r[0]]
        if ciertos:
            return True, max(r[1] for r in ciertos), [e for r in ciertos for e in r[2]]
        return False, 0.0, [e for r in res for e in r[2]]
    if t in ('umbral', 'no_umbral'):
        m = p[1]
        ctx.consultados.update(CAMPOS[m])
        u, v = buscar_umbral(ctx.app, m), medida(m, ctx.med)
        if m == 'SenalDebil' and ctx.med.get('conexion') != 'wifi':
            return False, 0.0, [f"{ETIQUETA[m]}: no aplica a conexión {ctx.med.get('conexion')}"]
        if u is None:
            return False, 0.0, [f"{ETIQUETA[m]}: no hay umbral definido para {ctx.app}"]
        if v is None:
            ctx.faltantes.add(m)
            return False, 0.0, [f"{ETIQUETA[m]}: dato no disponible"]
        valor, op, fuente = u
        origen = 'general' if (ctx.app, m) not in UMBRALES else ctx.app
        viola = v > valor if op == '>' else v < valor
        txt = f"{ETIQUETA[m]} = {v:.2f}{UNIDAD[m]}" if m == 'AnchoBajo' else f"{ETIQUETA[m]} = {v:g}{UNIDAD[m]}"
        if (t == 'umbral') == viola:
            rel = op if viola else ('<=' if op == '>' else '>=')
            return True, 1.0, [f"{txt} {rel} {valor}{UNIDAD[m]} (umbral {origen}, {fuente})"]
        return False, 0.0, [f"{txt}: {'no supera' if t == 'umbral' else 'supera'} el umbral {valor}{UNIDAD[m]}"]
    if t == 'hecho':
        return consulta(p[1])
    if t == 'no_hecho':
        ok, _, _ = consulta(p[1])
        if ok:
            return False, 0.0, [f"se verifica {fmt(p[1])}"]
        return True, 1.0, [f"no se verifica {fmt(p[1])}"]
    if t == 'flag':
        txt = f"{ctx.picos} de las últimas {ctx.total} mediciones con pérdida > {PICO_PERDIDA:g} %"
        return (True, 1.0, [txt]) if ctx.intermitente else (False, 0.0, [txt + " (insuficiente)"])
    if t == 'horario':
        ctx.consultados.add('horario')
        ok = ctx.med.get('horario') == p[1]
        return ok, (1.0 if ok else 0.0), [f"horario = {ctx.med.get('horario')}"]
    raise ValueError(f"Premisa desconocida: {p}")


def encadenar_adelante(ctx: Contexto):
    memoria: Dict[tuple, float] = {}
    traza = []

    def consulta(h):
        if h in memoria:
            return True, memoria[h], [f"{fmt(h)} (CF {memoria[h]:.2f})"]
        return False, 0.0, [f"no se verifica {fmt(h)}"]

    for estrato in (1, 2, 3):
        for r in (x for x in REGLAS if x.estrato == estrato):
            ok, cf_p, ev = evaluar(r.premisa, ctx, consulta)
            if ok:
                cf = r.cf * cf_p
                memoria[r.conclusion] = combinar(memoria[r.conclusion], cf) if r.conclusion in memoria else cf
                traza.append((r, cf, ev))
    return memoria, traza


def demostrar(meta: tuple, ctx: Contexto, memo: Dict):
    if meta in memo:
        return memo[meta]
    cf_total, lineas, motivos = None, [], []
    for r in REGLAS:
        if r.conclusion != meta:
            continue
        ok, cf_p, ev = evaluar(r.premisa, ctx, lambda h: demostrar(h, ctx, memo))
        if ok:
            cf = r.cf * cf_p
            cf_total = cf if cf_total is None else combinar(cf_total, cf)
            lineas.append(f"[{r.id}] {r.justificacion}")
            lineas += ['   ' + e for e in ev]
        else:
            motivos += ev
    if cf_total is None:
        memo[meta] = (False, 0.0, motivos)
    else:
        memo[meta] = (True, cf_total,
                      [f"{fmt(meta)} (CF {cf_total:.2f}) porque:"] + ['   ' + l for l in lineas])
    return memo[meta]


# ---------------------------------------------------------------------------
# Agente
# ---------------------------------------------------------------------------
class AgenteConectividad:
    def __init__(self):
        self.ventanas: Dict[str, deque] = {}

    def registrar(self, usuario: str, med: Dict):
        self.ventanas.setdefault(usuario, deque(maxlen=VENTANA)).append(med)

    def contexto(self, usuario: str, app: str, med: Dict) -> Contexto:
        v = self.ventanas.get(usuario, [])
        picos = sum(1 for m in v if m.get('perdida', 0) > PICO_PERDIDA)
        return Contexto(usuario, app, med, picos, len(v))

    def diagnosticar(self, usuario: str, app: str, med: Dict, mostrar=True, registrar=True) -> Dict:
        if registrar:
            self.registrar(usuario, med)
        ctx = self.contexto(usuario, app, med)
        memoria, traza = encadenar_adelante(ctx)
        causas = sorted(((cf, h[1]) for h, cf in memoria.items() if h[0] == 'Causa'), reverse=True)
        sintomas = [h[1] for h in memoria if h[0] == 'Sintoma']
        mala = ('ExperienciaMala',) in memoria
        if ('Diagnostico', 'SinProblemas') in memoria:
            decision = 'EXPERIENCIA BUENA'
        elif mala and causas:
            decision = f"EXPERIENCIA DEGRADADA - causa probable: {causas[0][1]}"
        elif mala:
            decision = 'REQUIERE_REVISION_MANUAL (síntomas sin causa identificada)'
        else:
            decision = 'SIN DEGRADACIÓN PERCIBIDA (con alertas)'
        res = {'ctx': ctx, 'memoria': memoria, 'traza': traza, 'causas': causas,
               'sintomas': sintomas, 'decision': decision}
        if mostrar:
            self.informe(res)
        return res

    def informe(self, res: Dict):
        ctx, memoria = res['ctx'], res['memoria']
        print(f"Usuario: {ctx.usuario} | Aplicación: {ctx.app} | Ventana: {ctx.total}/{VENTANA}")
        print("Cadena de razonamiento:")
        for r, cf, ev in res['traza']:
            print(f"  [{r.id}] {fmt(r.conclusion)}  (CF {cf:.2f})")
            print(f"        {r.justificacion}")
            for e in ev:
                print(f"        - {e}")
        if ('ExperienciaMala',) in memoria:
            print(f"\nExperiencia degradada (CF combinado {memoria[('ExperienciaMala',)]:.2f})")
        print(f"\nDIAGNÓSTICO: {res['decision']}")
        for cf, causa in res['causas']:
            print(f"  Causa {causa}: certeza {cf:.0%}")
            print(f"    Recomendación: {REMEDIOS[causa]}")
        if ctx.faltantes:
            print(f"  Datos no disponibles: {', '.join(sorted(ETIQUETA[m] for m in ctx.faltantes))}")

    def verificar(self, usuario: str, app: str, med: Dict, hipotesis: tuple):
        """Encadenamiento hacia atrás: prueba una hipótesis consultando solo lo necesario."""
        ctx = self.contexto(usuario, app, med)
        ok, cf, lineas = demostrar(hipotesis, ctx, {})
        print(f"Hipótesis: {fmt(hipotesis)}")
        if ok:
            print("  CONFIRMADA")
            for l in lineas:
                print("  " + l)
        else:
            print("  NO SE PUEDE PROBAR. Motivos:")
            for l in lineas:
                print(f"    - {l}")
        print(f"  Mediciones consultadas: {sorted(ctx.consultados)}")
        return ok, cf


# ---------------------------------------------------------------------------
# Interfaz de usuario
# ---------------------------------------------------------------------------
def pedir_mediciones():
    def num(msg):
        r = input(f"{msg} (Enter si no la tiene): ").strip()
        return float(r) if r else None

    app = input("Aplicación (Videollamada/Juegos/Streaming/Navegacion): ").strip()
    med = {'conexion': input("Conexión (wifi/ethernet/movil): ").strip().lower()}
    med['lat_gateway'] = num("Ping al router (ms)")
    med['lat_internet'] = num("Ping a internet (ms)")
    med['jitter'] = num("Jitter (ms)")
    med['perdida'] = num("Pérdida de paquetes (%)")
    med['descarga'] = num("Descarga medida (Mbps)")
    med['plan'] = num("Plan contratado (Mbps)")
    med['dns'] = num("Tiempo de resolución DNS (ms)")
    if med['conexion'] == 'wifi':
        med['senal_wifi'] = num("Señal Wi-Fi (dBm, ej. -65)")
    med['incr_lat_carga'] = num("Aumento de latencia bajo carga (ms)")
    med['horario'] = input("Horario (pico/valle): ").strip().lower()
    return app, med


# ---------------------------------------------------------------------------
# Casos de ejecución
# ---------------------------------------------------------------------------
def titulo(t):
    print("\n" + "=" * 78 + f"\n {t}\n" + "=" * 78)


def precargar(agente, usuario, perdidas):
    for p in perdidas:
        agente.registrar(usuario, {'perdida': p})


if __name__ == "__main__":
    agente = AgenteConectividad()

    titulo("CASO 1 - Videollamada con señal Wi-Fi débil")
    caso1 = {'conexion': 'wifi', 'senal_wifi': -74, 'lat_gateway': 38, 'lat_internet': 140,
             'jitter': 48, 'perdida': 2.5, 'descarga': 22, 'plan': 100, 'dns': 30,
             'incr_lat_carga': 60, 'horario': 'valle'}
    agente.diagnosticar('ana', 'Videollamada', caso1)

    print("\n--- Consulta dirigida (encadenamiento hacia atrás) ---")
    agente.verificar('ana', 'Videollamada', caso1, ('Causa', 'WiFi'))
    print()
    agente.verificar('ana', 'Videollamada', caso1, ('Causa', 'ISP'))

    titulo("CASO 2 - Juegos en línea, falla del proveedor (con historial intermitente)")
    precargar(agente, 'marcos', [0, 3.5, 0, 2.1, 0, 4.0, 0.2, 3.1, 0, 2.8, 0, 1.9, 0, 0, 3.3,
                                 0, 2.4, 0, 0.1])
    caso2 = {'conexion': 'ethernet', 'lat_gateway': 3, 'lat_internet': 210, 'jitter': 55,
             'perdida': 4.2, 'descarga': 85, 'plan': 100, 'dns': 40,
             'incr_lat_carga': 70, 'horario': 'valle'}
    agente.diagnosticar('marcos', 'Juegos', caso2)

    titulo("CASO 3 - Streaming con la red en buen estado")
    caso3 = {'conexion': 'wifi', 'senal_wifi': -55, 'lat_gateway': 4, 'lat_internet': 22,
             'jitter': 6, 'perdida': 0.1, 'descarga': 92, 'plan': 100, 'dns': 25,
             'incr_lat_carga': 20, 'horario': 'valle'}
    agente.diagnosticar('lucia', 'Streaming', caso3)

    titulo("CASO 4 - Mismas mediciones, dos aplicaciones distintas")
    caso4 = {'conexion': 'ethernet', 'lat_gateway': 2, 'lat_internet': 40, 'jitter': 5,
             'perdida': 0, 'descarga': 90, 'plan': 100, 'dns': 380,
             'incr_lat_carga': 15, 'horario': 'valle'}
    print("\n>> Navegación web")
    agente.diagnosticar('pedro', 'Navegacion', caso4)
    print("\n>> Juegos en línea")
    agente.diagnosticar('pedro', 'Juegos', caso4, registrar=False)

    titulo("MÓDULO DE ADQUISICIÓN - Cambio de umbral sin modificar el motor")
    caso5 = dict(caso3, lat_internet=130, jitter=10)
    print("Umbral de latencia para Juegos = 100 ms")
    r = agente.diagnosticar('sofia', 'Juegos', caso5, mostrar=False)
    print(f"  RTT = 130 ms -> {r['decision']}")
    actualizar_umbral('Juegos', 'Latencia', 150, '>', 'propio')
    print("Umbral actualizado a 150 ms")
    r = agente.diagnosticar('sofia', 'Juegos', caso5, mostrar=False)
    print(f"  RTT = 130 ms -> {r['decision']}")

    if MODO_INTERACTIVO:
        titulo("CONSULTA INTERACTIVA")
        app, med = pedir_mediciones()
        agente.diagnosticar('usuario', app, med)


 CASO 1 - Videollamada con señal Wi-Fi débil
Usuario: ana | Aplicación: Videollamada | Ventana: 1/20
Cadena de razonamiento:
  [R1] Sintoma(Jitter)  (CF 0.90)
        La variación de latencia supera lo tolerable para la aplicación
        - jitter = 48 ms > 30 ms (umbral Videollamada, norma)
  [R2] Sintoma(Perdida)  (CF 0.90)
        La pérdida de paquetes supera lo tolerable para la aplicación
        - pérdida = 2.5 % > 1 % (umbral Videollamada, norma)
  [R4] Sintoma(SenalDebil)  (CF 0.80)
        La señal Wi-Fi está por debajo del mínimo para la aplicación
        - señal Wi-Fi = -74 dBm < -67 dBm (umbral Videollamada, guia)
  [R6] Sintoma(AnchoBajo)  (CF 0.85)
        La velocidad obtenida es inferior a la proporción esperada del plan
        - descarga/plan = 0.22 < 0.5 (umbral Videollamada, propio)
  [R8] Causa(WiFi)  (CF 0.68)
        Señal débil acompañada de degradación: el problema está en el enlace inalámbrico
        - Sintoma(SenalDebil) (CF 0.80)
        - Sintoma(Jitter